# SIMPEL: winter climate and sensitivity comparison

This notebook compares the dry hourly forcing (`example_hourly_winter/`) with its rainfall scenario (`example_hourly_winter_rain/`). The meteorology is a **synthetic, Germany-like January scenario**, not observed data: mean air temperature 2 °C, mean daily KDOWN peak around 140 W/m², LDOWN around 290–295 W/m², and wind around 4.5 m/s at 10 m. Rain events in the wet scenario are 10, 20, and 40 mm at noon on 15–17 January.

Run the sensitivity analysis for both scenarios before plotting:

```sh
./sensitivity.sh example_hourly_winter/
./sensitivity.sh example_hourly_winter_rain/
```

The first table lists every parameter value used in each sensitivity run. The figures then compare precipitation, monthly water-balance summaries, overall response ranges by parameter, and response curves for ETaTotal, Runoff_total, and Seepage.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.titlesize": 18,
})
import re
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PROJECT_ROOT_OVERRIDE = None  # Set a path here if automatic project detection is ambiguous.


def find_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    current = Path.cwd()
    for candidate in (current, *current.parents):
        if (candidate / "sensitivity.sh").is_file():
            return candidate
    child_projects = [candidate for candidate in current.iterdir()
                      if candidate.is_dir() and (candidate / "sensitivity.sh").is_file()]
    if len(child_projects) == 1:
        return child_projects[0]
    raise FileNotFoundError(
        "Could not identify the project folder. Set PROJECT_ROOT_OVERRIDE to the folder containing sensitivity.sh."
    )


PROJECT_ROOT = find_project_root()
SCENARIOS = {"No rain": "example_hourly_winter", "Rain event": "example_hourly_winter_rain"}
SCENARIO_COLORS = {"No rain": "#E69F00", "Rain event": "#0072B2"}  # orange, blue
LEVEL_ORDER = ["minimum", "midpoint_low", "default", "midpoint_high", "maximum"]
LEVEL_LABELS = ["Minimum", "Mid-low", "Default", "Mid-high", "Maximum"]
SENSITIVITY_METRICS = ["ETaTotal", "Runoff_total", "Seepage"]


def latest_output_file(input_dir, filename):
    candidates = [
        folder / filename
        for folder in input_dir.iterdir()
        if folder.is_dir()
        and re.fullmatch(r"output_\d{8}_\d{6}", folder.name)
        and (folder / filename).is_file()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No {filename} found under timestamped output folders in {input_dir}. "
            "Run the model and sensitivity script first."
        )
    # Timestamped folder names sort chronologically; file mtimes can change when files are copied.
    return max(candidates, key=lambda path: path.parent.name)


def parse_hourly_dates(series):
    return pd.to_datetime(series, format="%d.%m.%Y.%H", errors="coerce")


forcing = {}
model_outputs = {}
sensitivity = {}
monthly_tables = []
for scenario, folder in SCENARIOS.items():
    input_dir = PROJECT_ROOT / folder
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Scenario input directory not found: {input_dir}")

    forcing_data = pd.read_csv(input_dir / "Global_Input.txt", sep="\t")
    forcing_data["DateTime"] = parse_hourly_dates(forcing_data["Date"])
    if forcing_data["DateTime"].isna().any():
        raise ValueError(f"Could not parse every hourly date in {input_dir / 'Global_Input.txt'}")
    forcing[scenario] = forcing_data

    model_path = latest_output_file(input_dir, "SIMPLE_bucket_model.csv")
    model_data = pd.read_csv(model_path, sep="\t")
    model_data["DateTime"] = parse_hourly_dates(model_data["Date"])
    if model_data["DateTime"].isna().any():
        raise ValueError(f"Could not parse every model output date in {model_path}")
    model_outputs[scenario] = model_data

    sensitivity_path = latest_output_file(input_dir, "sensitivity_oat.csv")
    sensitivity_data = pd.read_csv(sensitivity_path, sep="\t")
    required = {"parameter", "level", "value", "note"}
    missing = required.difference(sensitivity_data.columns)
    if missing:
        raise ValueError(f"{sensitivity_path} is missing columns: {sorted(missing)}")
    sensitivity[scenario] = sensitivity_data

    model_data["Month"] = model_data["DateTime"].dt.to_period("M").astype(str)
    flux_columns = [name for name in [
        "Precipitation", "ETaTotal", "ETa", "I-Leaf", "I-Litter", "Seepage",
        "Surface_runoff", "Runoff_total",
    ] if name in model_data.columns]
    monthly = model_data.groupby("Month", sort=True)[flux_columns].sum()
    if "Storage" in model_data.columns:
        monthly["Storage_end"] = model_data.groupby("Month", sort=True)["Storage"].last()
    monthly.insert(0, "Condition", scenario)
    monthly_tables.append(monthly.reset_index())

monthly_summary = pd.concat(monthly_tables, ignore_index=True)
parameter_order = [
    parameter for parameter in sensitivity["No rain"]["parameter"].drop_duplicates()
    if parameter.lower() != "base"
]
reference_values = None
for scenario, table in sensitivity.items():
    changed = table.loc[table["parameter"].str.lower() != "base"].copy()
    missing_levels = {
        parameter: sorted(set(LEVEL_ORDER).difference(group["level"]))
        for parameter, group in changed.groupby("parameter")
        if set(LEVEL_ORDER).difference(group["level"])
    }
    if missing_levels:
        raise ValueError(f"{scenario} is missing sensitivity levels: {missing_levels}")
    if set(changed["parameter"]) != set(parameter_order):
        raise ValueError(f"Parameter set differs between scenarios in {scenario}.")
    values = changed.pivot(index="parameter", columns="level", values="value")
    values = values.reindex(index=parameter_order, columns=LEVEL_ORDER)
    if values.isna().any().any():
        raise ValueError(f"{scenario} has missing parameter values.")
    if reference_values is None:
        reference_values = values
    elif not np.allclose(values.to_numpy(dtype=float), reference_values.to_numpy(dtype=float)):
        raise ValueError("Parameter values differ between scenarios.")

variable_values = pd.DataFrame({
    "Parameter": parameter_order,
    "Default": reference_values["default"].to_numpy(),
    "Min / Max": [
        f"{minimum:g} / {maximum:g}"
        for minimum, maximum in zip(reference_values["minimum"], reference_values["maximum"])
    ],
})
available_metrics = [
    metric for metric in SENSITIVITY_METRICS
    if all(f"{metric}_change_%" in table.columns for table in sensitivity.values())
]
if not available_metrics:
    raise ValueError("No selected sensitivity change columns were found in both scenarios.")

print(f"Project: {PROJECT_ROOT}")
print("Loaded forcing, model outputs and sensitivity tables for both conditions.")

Project: /mnt/c/Users/hswt603qia/IdeaProjects/Simpel
Loaded forcing, model outputs and sensitivity tables for both conditions.


## Parameter values used

This table shows each parameter's default value and tested minimum / maximum. The combined LAI row is the value assigned to **both** `MinLAI` and `MaxLAI`.

In [2]:
display(variable_values.fillna(""))


,Parameter,Default,Min / Max
0,Root Depth,100.0000,1 / 10000
1,Field Capacity %,25.0000,12 / 100
2,Permanent Wilting Point %,12.0000,0 / 25
3,Init-Value Soil %,24.0000,0 / 25
4,Glugla coeff.,150.0000,0 / 1e+06
5,Infiltration fraction,0.2500,0 / 1
6,Land Use Fraction,0.7500,0 / 1
7,Layer Thickness,0.3500,0 / 10
8,Max. Drainage Rate,2.8800,0 / 1000
9,Drainage Coeff. b,3.7000,0 / 100


## Forcing precipitation and monthly water-balance summary

Figure 1 shows the hourly precipitation series for the two conditions. The monthly table summarizes model fluxes and end-of-month soil storage.

In [3]:
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(14, 4), sharey=True)
for ax, (scenario, forcing_data) in zip(axes, forcing.items()):
    ax.step(forcing_data["DateTime"], forcing_data["P"], where="mid", linewidth=1.1,
            color=SCENARIO_COLORS[scenario])
    ax.set_title(scenario)
    ax.set_xlabel("Date")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Precipitation (mm per hourly timestep)")
fig.suptitle("Figure 1. Hourly forcing precipitation", y=1.02)
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
plt.show()

monthly_summary.round(2)

/tmp/ipykernel_217224/3669454649.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Month,Condition,Precipitation,ETaTotal,ETa,I-Leaf,I-Litter,Seepage,Surface_runoff,Runoff_total,Storage_end
0,2004-01,No rain,0.0,5.31,5.31,0.0,0.0,41.17,0.00,41.17,193.52
1,2004-01,Rain event,70.0,6.44,5.15,1.3,0.0,41.80,66.89,108.69,194.87


## Sensitivity overview by parameter

Figure 2 is a heatmap of signed percentage changes across all five tested levels. Rows compare the no-rain and rain-event forcings; columns compare ETaTotal, Runoff_total, and Seepage. Each metric has a shared color scale across both forcing conditions, with zero at the center.


In [4]:
fig, axes = plt.subplots(
    2, len(available_metrics), figsize=(7 * len(available_metrics), 13),
    squeeze=False, sharey=True,
)
matrices_by_metric = {}

for column, metric in enumerate(available_metrics):
    change_column = f"{metric}_change_%"
    matrices = []
    for scenario in SCENARIOS:
        changed = sensitivity[scenario]
        changed = changed.loc[changed["parameter"].str.lower() != "base"]
        matrix = changed.pivot(index="parameter", columns="level", values=change_column)
        matrix = matrix.reindex(index=parameter_order, columns=LEVEL_ORDER)
        values = matrix.to_numpy(dtype=float)
        matrices.append(values)
    matrices_by_metric[metric] = matrices

# Values outside the fixed range use solid dark endpoint colors.
norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
cmap = plt.get_cmap("coolwarm").copy()
cmap.set_under("#08306b")
cmap.set_over("#67000d")

for column, metric in enumerate(available_metrics):
    for row, (scenario, matrix) in enumerate(zip(SCENARIOS, matrices_by_metric[metric])):
        ax = axes[row, column]
        image = ax.imshow(matrix, aspect="auto", cmap=cmap, norm=norm,
                          interpolation="nearest")
        ax.set_title(f"{scenario}: {metric}")
        ax.set_xticks(np.arange(len(LEVEL_ORDER)), LEVEL_LABELS, rotation=25, ha="right")
        if column == 0:
            ax.set_yticks(np.arange(len(parameter_order)), parameter_order)
            ax.set_ylabel("Changed parameter")
        else:
            ax.tick_params(axis="y", labelleft=False)
        ax.set_xlabel("Test level")

fig.suptitle("Figure 2. Sensitivity overview by parameter and test level", y=0.985)
# Reserve a right-hand strip for one colorbar shared by every subplot.
fig.subplots_adjust(left=0.22, right=0.89, bottom=0.09, top=0.94,
                    wspace=0.35, hspace=0.28)
colorbar_axis = fig.add_axes([0.92, 0.18, 0.018, 0.66])
colorbar = fig.colorbar(image, cax=colorbar_axis, extend="both", ticks=[-100, 0, 100])
colorbar.set_label("Change from condition default (%)", fontsize=14, labelpad=12)
colorbar.ax.tick_params(labelsize=13)
plt.show()


/tmp/ipykernel_217224/3204589081.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Output response at each tested parameter value

Figures 3–5 show each metric against the tested value for every parameter. Each parameter is varied independently; both LAI limits are assigned the same tested value. Scenario colors are consistent throughout: orange for no rain and blue for the rain events.

In [1]:
metric_titles = [
    ("ETaTotal", "total actual evapotranspiration"),
    ("Runoff_total", "total runoff"),
    ("Seepage", "seepage"),
]
columns = 4
rows = int(np.ceil(len(parameter_order) / columns))

for figure_number, (metric, description) in enumerate(metric_titles, start=3):
    change_column = f"{metric}_change_%"
    if not all(change_column in table.columns for table in sensitivity.values()):
        print(f"Skipping {metric}: its change column is missing.")
        continue
    fig, axes = plt.subplots(rows, columns, figsize=(20, 3.5 * rows), squeeze=False)
    for ax, parameter in zip(axes.ravel(), parameter_order):
        for scenario, table in sensitivity.items():
            subset = table.loc[table["parameter"] == parameter].copy()
            subset["level"] = pd.Categorical(subset["level"], LEVEL_ORDER, ordered=True)
            subset = subset.sort_values("level")
            ax.plot(subset["value"], subset[change_column], marker="o", markersize=4,
                    linewidth=1.2, color=SCENARIO_COLORS[scenario], label=scenario)
        ax.axhline(0, color="black", linewidth=0.7)
        ax.set_title(parameter, fontsize=11)
        ax.set_xlabel("Tested value")
        ax.set_ylabel(f"{metric} change (%)")
        ax.grid(alpha=0.2)
    for ax in axes.ravel()[len(parameter_order):]:
        ax.set_visible(False)
    handles = [Line2D([0], [0], color=SCENARIO_COLORS[name], marker="o", label=name)
               for name in SCENARIOS]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.008),
               ncol=len(handles), frameon=False, title="Forcing condition")
    fig.suptitle(f"Figure {figure_number}. {metric} response ({description})", y=0.99)
    fig.tight_layout(rect=(0, 0.035, 1, 0.97))
    plt.show()

NameError: name 'np' is not defined